# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** Each row represents one item in an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

# Keep order and vendor_id on every row
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

# Assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

# Left join
joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

# Report which vendor did not match and how many units are attached to it
unmatched = joined[joined['_merge'] == 'left_only']

print('Unmatched Vendors:', unmatched) # Empty dataframe because there were no vendors that did not match
print('Umatched Units:', unmatched['qty'].sum())

Unmatched Vendors: Empty DataFrame
Columns: [item, qty, order, vendor_id, name, zone, _merge]
Index: []
Umatched Units: 0


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Because left join, unmatched vendor would have NaN as name
quantity_per_vendor = joined.groupby('name', dropna=False)['qty'].sum()
print(quantity_per_vendor)

# There are no unmatched vendors in this table

name
Cav Merch       1
Hoos Burgers    6
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_items = joined.groupby('zone')['qty'].sum()

print(zone_items)
print("Zone item totals:", zone_items.sum())

# Zone A sold the most items - 7. The total number of Zone items is 7. Therefore, the difference is 0.

zone
A    7
Name: qty, dtype: int64
Zone item totals: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Pull out orders with lines
orders_with_lines = []
for order in ragged_json:
    if 'lines' in order:
        orders_with_lines.append(order)

# Flatten
lines_updated = pd.json_normalize(
    orders_with_lines, # Will not crash
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(lines_updated)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

"One row per line item" is a useful shape because every item and its quantity can be analyzed separately. A question that would become easier to answer with this way of thinking is "What is the total quantity of items sold?".

Missing quantities mean that the data was not provided, while a quantity of zero means that the data was provided and the item was not sold. If you treat missing quantities like a quantity of zero, your analysis will not be accurate. For instance, the total quantity sold may be lower than it actually is if there are many missing quanitities.